# Portfolio Risk Management: Factor Models, Stress Testing and Crisis Hedging

This notebook covers the full workflow of:
* factor-model-driven portfolio construction
* stress testing under two geopolitical crises
* risk-based rebalancing

**Two crisis tracks:**
- Gulf Crisis: oil spike, stagflation, EM stress
- Ukraine/Russia: energy and food shock, defense surge, European risk premium

**Workflows:**
1. BUILD: Build factor model and benign portfolio
2. BUILD: Define crisis portfolios for each scenario
3. RISK: Stress test benign portfolio against both crises
4. RISK: Hedge and rebalance
5. ANALYSIS: Compare outcomes

---
## Part 1: Factor Model Foundations

### 1.1 Basics

A factor model decomposes portfolio returns into:
- **Systematic returns**: driven by common risk factors
- **Idiosyncratic returns**: asset-specific, diversifiable

$$
r_i = \alpha_i + \sum_{k=1}^{K} \beta_{ik} f_k + \epsilon_i
$$

where:
- $r_i$ = return of asset $i$
- $\beta_{ik}$ = exposure (loading) of asset $i$ to factor $k$
- $f_k$ = return of factor $k$
- $\epsilon_i$ = idiosyncratic return, $\epsilon_i \sim \mathcal{N}(0, \sigma_i^2)$

Portfolio return:
$$
r_p = \sum_i w_i r_i = \sum_{k} \left(\sum_i w_i \beta_{ik}\right) f_k + \sum_i w_i \epsilon_i = \sum_k B_{pk} f_k + \epsilon_p
$$

Portfolio variance:
$$
\sigma_p^2 = \mathbf{B}_p^\top \mathbf{F} \mathbf{B}_p + \mathbf{w}^\top \mathbf{\Delta} \mathbf{w}
$$

where $\mathbf{F}$ is the factor covariance matrix and $\mathbf{\Delta}$ is the diagonal idiosyncratic variance matrix.

### 1.2 Factor Selection

For a multi-asset portfolio spanning equities, rates, and geopolitical risks, we use three layers of factors:

**Macro factors** — drive cross-asset returns

| Factor | Proxy | Rationale |
|---|---|---|
| Oil price | Brent crude return | Key driver of inflation, EM, energy sector |
| Inflation surprise | 5Y breakeven rate change | Reprices bonds and rate-sensitive equities |
| Real rates | 5Y TIPS yield change | Core driver of bond and growth equity valuations |
| USD strength | DXY return | EM stress, commodity prices, risk-off |
| Credit spread | IG/HY OAS change | Risk appetite, funding conditions |
| Geopolitical risk | GPR index change | Tail risk, defense, safe havens |
| Food/agriculture | Bloomberg Agri index | Ukraine-specific transmission |

where: 
* **DXY**: US Dollar Index acroos basket of major currencies (EUR, JPY, GBP, CAD, SEK, CHF)
* **OAS**: Option Adjusted Spread
* **GPR**: Geopol risk index: uses newspaper text analysis

**Style factors** — drive cross-sectional equity returns

| Factor | Definition | Rationale |
|---|---|---|
| Momentum | 12M-1M price return | Trend following, crisis amplification |
| Value | B/P ratio | Mean reversion, cheap vs expensive |
| Quality | ROE, low leverage | Defensive in stress |
| Size | Market cap | Small cap more vulnerable in risk-off |
| Low volatility | Realized vol | Defensive factor |

**Sector/regional factors** — capture industry and geography

| Factor | Relevance |
|---|---|
| Energy | Gulf and Ukraine direct exposure |
| Defense | Ukraine specific |
| Tourism/hospitality | Gulf specific |
| EM Asia | Growth factor, oil importer stress |
| European equities | Ukraine proximity premium |
| Insurance | Rate sensitivity, cat risk |

### 1.3 Factor Covariance Matrix

In practice, $\mathbf{F}$ is estimated from historical returns using a shrinkage estimator to avoid overfitting:

$$
\hat{\mathbf{F}} = (1 - \delta) \mathbf{S} + \delta \mathbf{T}
$$

where $\mathbf{S}$ is the sample covariance, $\mathbf{T}$ is the shrinkage target (e.g. constant correlation), and $\delta$ is the shrinkage intensity estimated via Ledoit-Wolf.

Factor exposures $\beta_{ik}$ are estimated via time-series OLS for macro factors, and cross-sectional regression for style factors.

In [1]:
from quant_risk.setup import base, macro

np, pd, plt = base()
fed_client, store, external_store = macro()

base loaded
macro loaded


In [2]:
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
if False:
    store.clear_cache()   
    external_store.clear_cache()

cleared 10 files from cache
cleared 17 files from cache


In [12]:
fred_series = [
    "US10Y",
    "US2Y",
    "VIX",
    "IG_OAS",
    "HY_OAS",
    "BRENT",
    "BREAKEVEN5Y",
    "REAL5Y",
    "EURUSD",
    "DXY",
]

df_fred = store.build_panel(fred_series)
df_fred["Curve_Slope"] = df_fred["US10Y"] - df_fred["US2Y"]
print(df_fred.shape)
df_fred.isna().sum()

(16372, 11)


US10Y            303
US2Y            3895
VIX             7193
IG_OAS         15588
HY_OAS         15587
BRENT           6489
BREAKEVEN5Y    10533
REAL5Y         10534
EURUSD          9518
DXY            11275
Curve_Slope     3895
dtype: int64

In [13]:
ff = external_store.get_fama_french(frequency='daily')

style_factors = pd.DataFrame({
    'momentum': ff['Mom'],
    'value':    ff['HML'],
    'quality':  ff['RMW'],
    'size':     ff['SMB'],
    'low_vol':  -ff['Mkt-RF'],
}).dropna()

print(style_factors.tail(2))
print(style_factors.shape)

            momentum     value   quality      size   low_vol
2026-03-30 -0.027500  0.004400  0.009400 -0.005900  0.004400
2026-03-31  0.026500 -0.016500 -0.014800  0.003400 -0.029800
(6600, 5)


In [14]:
yf_series = [
    "ENERGY_SECTOR",
    "DEFENSE_SECTOR",
    "EUROPE_EQ",
    "EM_ASIA",
    "INSURANCE",
    "TOURISM",
    "GLD",
    "TLT",
    "EEM",
    "IEF",
    "LQD",
    "HYG",
    "EWG",
    "EWI",
]

df_yf = external_store.build_panel(yf_series)
print(df_yf.shape)
df_yf.isna().sum()

(6625, 14)


ENERGY_SECTOR        0
DEFENSE_SECTOR    1593
EUROPE_EQ          702
EM_ASIA           1814
INSURANCE         1593
TOURISM           1375
GLD               1226
TLT                644
EEM                822
IEF                644
LQD                644
HYG               1826
EWG                  0
EWI                  0
dtype: int64

In [15]:
# GPR
gpr = external_store.get_gpr()
gpr.tail(2)

Date
2026-04-29   171.760000
2026-04-30   171.160000
Name: GPR, dtype: float64

In [18]:
# ── Factor assembly ───────────────────────────────────────────────────────────
# Convert all price series to returns and align to common index

# FRED -- rates and spreads are already levels, take diff for changes
# prices take pct_change
macro_factors = pd.DataFrame({
    'oil':            df_fred['BRENT'].pct_change(),
    'inflation_surp': df_fred['BREAKEVEN5Y'].diff(),
    'real_rates':     df_fred['REAL5Y'].diff(),
    'usd':            df_fred['DXY'].pct_change(),
    # 'credit_spread':  df_fred['HY_OAS'].diff(),
    'credit_spread':  df_yf['HYG'].pct_change(),  # HY ETF return as credit proxy
    'geo_risk':       gpr.pct_change(),
    'agriculture':    df_fred['BRENT'].pct_change(),  # proxy, see note
    'curve_slope':    df_fred['Curve_Slope'].diff(),
    'vix_change':     df_fred['VIX'].diff(),
})

# Style factors -- already in return space from French library
style_factors = pd.DataFrame({
    'momentum': ff['Mom'],
    'value':    ff['HML'],
    'quality':  ff['RMW'],
    'size':     ff['SMB'],
    'low_vol':  -ff['Mkt-RF'],
})

# Sector/regional -- price ETFs to returns
sector_factors = pd.DataFrame({
    'energy_sector':  df_yf['ENERGY_SECTOR'].pct_change(),
    'defense_sector': df_yf['DEFENSE_SECTOR'].pct_change(),
    'europe_eq':      df_yf['EUROPE_EQ'].pct_change(),
    'em_asia':        df_yf['EM_ASIA'].pct_change(),
    'insurance':      df_yf['INSURANCE'].pct_change(),
    'tourism':        df_yf['TOURISM'].pct_change(),
})

# Safe havens -- useful for crisis analysis
safe_havens = pd.DataFrame({
    'gold':  df_yf['GLD'].pct_change(),
    'bonds': df_yf['TLT'].pct_change(),
})

# ── Align all to common index ─────────────────────────────────────────────────
factor_df = pd.concat(
    [macro_factors, style_factors, sector_factors, safe_havens],
    axis=1,
)

# drop to common non-null start -- 2008 where coverage is fullest
factor_df = factor_df['2008-01-01':]
factor_df = factor_df.dropna()

print(f"Factor panel shape: {factor_df.shape}")
print(f"Date range: {factor_df.index[0].date()} to {factor_df.index[-1].date()}")
print(f"Factors: {factor_df.columns.tolist()}")

Factor panel shape: (4291, 22)
Date range: 2008-01-03 to 2026-03-31
Factors: ['oil', 'inflation_surp', 'real_rates', 'usd', 'credit_spread', 'geo_risk', 'agriculture', 'curve_slope', 'vix_change', 'momentum', 'value', 'quality', 'size', 'low_vol', 'energy_sector', 'defense_sector', 'europe_eq', 'em_asia', 'insurance', 'tourism', 'gold', 'bonds']
